# Single-grain coercivity results

Load all `.npz` outputs produced by `single_grain_coercivity.py` or `run_script.sh` from the local `results/` directory, collect coercivity metadata, and plot coercivity versus grain size.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


In [ ]:
RESULTS_DIR = Path('results')
RESULT_PATTERN = 'single_grain_*.npz'

result_files = sorted(RESULTS_DIR.glob(RESULT_PATTERN))
print(f'Found {len(result_files)} result file(s) in {RESULTS_DIR.resolve()}')
for path in result_files:
    print(path)


In [ ]:
def scalar(data, key, default=np.nan) -> object:
    if key not in data.files:
        return default
    value = data[key]
    if np.asarray(value).shape == ():
        return value.item()
    return value

rows = []
for path in result_files:
    with np.load(path, allow_pickle=True) as data:
        rows.append({
            'file': path.name,
            'L_m': float(scalar(data, 'L')),
            'size_factor': float(scalar(data, 'size_factor')),
            'n': int(scalar(data, 'n')),
            'ntot': int(scalar(data, 'ntot')),
            'use_fmm': bool(scalar(data, 'use_fmm', False)),
            'Hc_A_per_m': float(scalar(data, 'Hc_A_per_m', scalar(data, 'Hc'))),
            'Hc_T': float(scalar(data, 'Hc_T')),
            'runtime_s': float(scalar(data, 'runtime')),
        })

rows = sorted(rows, key=lambda row: (row['use_fmm'], row['size_factor'], row['n']))
print('file\tsize_factor\tL_m\tn\tntot\tuse_fmm\tHc_A_per_m\tHc_T\truntime_s')
for row in rows:
    print(
        f"{row['file']}\t{row['size_factor']:.6g}\t{row['L_m']:.6e}\t"
        f"{row['n']}\t{row['ntot']}\t{row['use_fmm']}\t"
        f"{row['Hc_A_per_m']:.6e}\t{row['Hc_T']:.6e}\t{row['runtime_s']:.3f}"
    )


In [ ]:
if rows:
    fig, ax = plt.subplots(figsize=(8, 5))
    resolutions = sorted({row['n'] for row in rows})
    for n in resolutions:
        subset = [row for row in rows if row['n'] == n]
        x = [row['size_factor'] for row in subset]
        y = [row['Hc_T'] for row in subset]
        ax.plot(x, y, '.-', label=f'n={n}')

    ax.set_xlabel('grain size / characteristic length [-]')
    ax.set_ylabel('coercivity $\mu_0 H_c$ [T]')
    ax.set_title('Single-grain coercivity sweep')
    ax.grid(True, linestyle='--', linewidth=0.5)
    ax.legend(title='resolution')
    fig.tight_layout()
else:
    print(
        'No result files found. Run ./run_script.sh or '
        'single_grain_coercivity.py first.'
    )
